# 04 — BTM: MIMIC-III in-hospital mortality

Dataset-specific experiment entry point for the BTM stage. The reusable BTM optimisation is in `btm/utils.py`; this notebook keeps the paper-facing hyperparameters, data/model choice, logging and evaluation explicit.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "btm").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from btm.loaders import DATASET_CHOICES, load_dataset
from btm.models import default_net_type, get_model
from btm.utils import *


## Configuration


In [ ]:
DATASET = "mimic3_ihm"
NET_TYPE = "tcn2"
OPTIM = "sgd"
IPC = 100
INIT = "real"          # "real", "rand", "real_soft"
LEARN_Y = False
SEED = 42
TRAIN_BATCH = 256
USE_WANDB = False

btm_cfg = BTMConfig(
    synthetic_budget=IPC,
    iterations=10_000,
    expert_window=0.20,
    syn_steps=30,
    batch_syn=2 * IPC,
    eval_every=10,
    learn_y=LEARN_Y,
    init=INIT,
    sequential_generation=False,
    compute_residual=True,
)


## Load data, model factory and learned mode connections


In [ ]:
set_seed(SEED)
device = get_device()
bundle = load_dataset(
    DATASET,
    data_root=ROOT / "data",
    train_batch=TRAIN_BATCH,
    pre_process="std",
)
train_loader, val_loader, test_loader = (
    bundle.train_loader, bundle.val_loader, bundle.test_loader
)

model_factory = lambda: get_model(
    DATASET, NET_TYPE, device=device, output_dim=bundle.output_dim
)

paths = experiment_paths(DATASET, NET_TYPE, OPTIM, root=ROOT)
modes = torch.load(paths["modes"], map_location="cpu", weights_only=False)
print(f"loaded {len(modes)} modes from {paths['modes']}")

pretrained_model = None
if bundle.task == "binary" and INIT == "real_soft":
    pretrained_model = model_factory()
    pretrained_model.load_state_dict(modes[0][1])


## Evaluation hyperparameters


In [ ]:
eval_lr, eval_momentum, eval_epochs = get_training_hyperparameters(
    DATASET, OPTIM, stage="btm_eval"
)
print(eval_lr, eval_momentum, eval_epochs)


## Optional W&B logging


In [ ]:
logger = None
wandb_run = None
if USE_WANDB:
    import wandb
    # Authenticate outside the notebook (`wandb login` or WANDB_API_KEY).
    wandb_run = wandb.init(
        project="BTM-MIMIC3-IHM",
        name=f"{NET_TYPE}_{OPTIM}_ipc{IPC}_{INIT}",
        config=btm_cfg.__dict__,
    )
    logger = wandb.log


## Run BTM


In [ ]:
save_dir = ROOT / "stored" / "BTM" / DATASET / f"{NET_TYPE}_{OPTIM}"
save_path = save_dir / f"{INIT}_{IPC}.pth"

result = run_btm(
    model_factory=model_factory,
    train_loader=train_loader,
    val_loader=val_loader,
    modes=modes,
    task=bundle.task,
    device=device,
    config=btm_cfg,
    eval_lr=eval_lr,
    eval_momentum=eval_momentum,
    eval_epochs=eval_epochs,
    optim=OPTIM,
    save_path=save_path,
    seed=SEED,
    pretrained_model=pretrained_model,
    logger=logger,
)
print("best validation APR:", result.best_val_apr)
print("saved:", result.save_path)

if wandb_run is not None:
    wandb_run.finish()


## Repeated final evaluation


In [ ]:
metrics = evaluate_synthetic_repeated(
    model_factory=model_factory,
    x_syn=result.best_x_syn,
    y_syn=result.best_y_syn,
    val_loader=val_loader,
    test_loader=test_loader,
    task=bundle.task,
    device=device,
    optim=OPTIM,
    lr=eval_lr,
    momentum=eval_momentum,
    epochs=eval_epochs,
    repeats=20,
)

for name, (mean, std) in metrics.items():
    print(f"{name:>8s}: {mean:.4f} ± {std:.4f}")
print("learned synthetic lr:", float(result.syn_lr))


## Training history


In [ ]:
if result.history:
    iterations = [row["iteration"] for row in result.history]
    val_apr = [row["val_apr"] for row in result.history]
    tm_loss = [row["tm_loss"] for row in result.history]

    fig, ax1 = plt.subplots(figsize=(8, 4))
    ax1.plot(iterations, val_apr, label="validation APR")
    ax1.set_xlabel("BTM iteration")
    ax1.set_ylabel("APR")
    ax1.grid(True)

    ax2 = ax1.twinx()
    ax2.plot(iterations, tm_loss, label="trajectory-matching loss")
    ax2.set_ylabel("TM loss")
    plt.show()
